## 1. Lendo o arquivo fonte

Nesta prática, vamos partir da mesma página fictícia usada no tamanho fixo. A fonte é um arquivo Markdown, e ainda não um conjunto de chunks. Primeiro vamos extrair o conteúdo original; depois vamos usar a estrutura que o próprio arquivo oferece.

In [2]:
from pathlib import Path

DATA_PATH = Path("../data/integracoes-resilientes-webhooks.md")

source_document = DATA_PATH.read_text(encoding="utf-8")

print(f"arquivo fonte: {DATA_PATH}")
print(f"caracteres no arquivo: {len(source_document)}")
print(source_document[:420] + "...")

arquivo fonte: ../data/integracoes-resilientes-webhooks.md
caracteres no arquivo: 1589
---
book_title: "Integracoes Resilientes: webhooks, filas e retentativas na pratica"
edition: "2a edicao"
chapter: "Capitulo 4 - Webhooks em producao"
section: "4.3 Timeouts e retentativas"
page_start: 118
page_end: 119
---

# 4.3 Timeouts e retentativas em webhooks

Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor.
Se a conexao expira antes de receber confirmacao, o provedor nao sabe...


## 2. Separando metadados e texto bruto

O frontmatter descreve a origem da página inteira. O corpo abaixo dele é o material que será analisado em blocos, sem perder a referência ao livro, à edição, ao capítulo e à seção.

In [4]:
def parse_frontmatter(markdown):
    if not markdown.startswith("---\n"):
        return {}, markdown.strip()

    _, frontmatter, body = markdown.split("---", 2)
    metadata = {}

    for line in frontmatter.strip().splitlines():
        key, value = line.split(":", 1)
        value = value.strip().strip('"')
        if value.isdigit():
            value = int(value)
        metadata[key.strip()] = value

    return metadata, body.strip()


page_metadata, raw_page = parse_frontmatter(source_document)

print(f"fonte: {page_metadata['book_title']}")
print(f"trecho: {page_metadata['chapter']} > {page_metadata['section']}")
print(f"paginas: {page_metadata['page_start']}-{page_metadata['page_end']}")
print(f"caracteres no texto bruto: {len(raw_page)}")

fonte: Integracoes Resilientes: webhooks, filas e retentativas na pratica
trecho: Capitulo 4 - Webhooks em producao > 4.3 Timeouts e retentativas
paginas: 118-119
caracteres no texto bruto: 1363


## 3. Extraindo blocos estruturais

Agora vamos reconhecer as fronteiras que o Markdown já preserva: heading, parágrafo, tabela e bloco de código. Cada bloco recebe também o caminho de headings que o localiza na hierarquia do documento.

In [6]:
import re

def parse_markdown_blocks(markdown):
    lines = markdown.splitlines()
    blocks = []
    heading_path = []
    index = 0

    while index < len(lines):
        line = lines[index]

        if not line.strip():
            index += 1
            continue

        heading = re.match(r"^(#{1,6})\s+(.*)$", line)
        if heading:
            level = len(heading.group(1))
            heading_path = heading_path[: level - 1] + [heading.group(2).strip()]
            blocks.append({
                "kind": "heading",
                "text": line,
                "heading_path": heading_path.copy(),
            })
            index += 1
            continue

        if line.startswith("```"):
            block_lines = [line]
            index += 1
            while index < len(lines) and not lines[index].startswith("```"):
                block_lines.append(lines[index])
                index += 1
            if index < len(lines):
                block_lines.append(lines[index])
                index += 1
            blocks.append({
                "kind": "code",
                "text": "\n".join(block_lines),
                "heading_path": heading_path.copy(),
            })
            continue

        if line.startswith("|"):
            block_lines = []
            while index < len(lines) and (lines[index].startswith("|") or not lines[index].strip()):
                if lines[index].strip():
                    block_lines.append(lines[index])
                index += 1
            blocks.append({
                "kind": "table",
                "text": "\n".join(block_lines),
                "heading_path": heading_path.copy(),
            })
            continue

        block_lines = [line]
        index += 1
        while (
            index < len(lines)
            and lines[index].strip()
            and not re.match(r"^#{1,6}\s+", lines[index])
            and not lines[index].startswith("|")
            and not lines[index].startswith("```")
        ):
            block_lines.append(lines[index])
            index += 1

        first_line = block_lines[0].lstrip()
        kind = "list" if first_line.startswith(("- ", "* ")) else "paragraph"
        blocks.append({
            "kind": kind,
            "text": "\n".join(block_lines),
            "heading_path": heading_path.copy(),
        })

    return blocks


structural_blocks = parse_markdown_blocks(raw_page)

print(f"blocos encontrados: {len(structural_blocks)}")
for position, block in enumerate(structural_blocks, start=1):
    path = " > ".join(block["heading_path"])
    print(f"{position:02d} | {block['kind']:<9} | {path}")

blocos encontrados: 7
01 | heading   | 4.3 Timeouts e retentativas em webhooks
02 | paragraph | 4.3 Timeouts e retentativas em webhooks
03 | paragraph | 4.3 Timeouts e retentativas em webhooks
04 | table     | 4.3 Timeouts e retentativas em webhooks
05 | paragraph | 4.3 Timeouts e retentativas em webhooks
06 | code      | 4.3 Timeouts e retentativas em webhooks
07 | paragraph | 4.3 Timeouts e retentativas em webhooks


## 4. Contando tokens com o tokenizer do modelo

A estrutura define os candidatos, mas o modelo ainda impõe um limite. Vamos usar o tokenizer real de `sentence-transformers/all-MiniLM-L6-v2`; não vamos gerar embeddings nesta prática. A contagem inclui os tokens especiais que o modelo recebe e desativa padding e truncamento para medir o texto inteiro.

In [8]:
from tokenizers import Tokenizer

TOKENIZER_NAME = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_MAX_TOKENS = 128
MAX_CHUNK_TOKENS = MODEL_MAX_TOKENS

tokenizer = Tokenizer.from_pretrained(TOKENIZER_NAME)
tokenizer.no_padding()
tokenizer.no_truncation()

def count_tokens(text):
    return len(tokenizer.encode(text).ids)


for block in structural_blocks:
    block["token_count"] = count_tokens(block["text"])

print(f"tokenizer: {TOKENIZER_NAME}")
print(f"limite do modelo: {MODEL_MAX_TOKENS} tokens")
for position, block in enumerate(structural_blocks, start=1):
    print(f"{position:02d} | {block['kind']:<9} | {block['token_count']:>3} tokens")

tokenizer: sentence-transformers/all-MiniLM-L6-v2
limite do modelo: 128 tokens
01 | heading   |  19 tokens
02 | paragraph |  94 tokens
03 | paragraph |  91 tokens
04 | table     | 126 tokens
05 | paragraph |  31 tokens
06 | code      |  67 tokens
07 | paragraph |  90 tokens


## 5. Compondo chunks estruturais dentro do limite

A composição é gulosa e local: percorremos os blocos na ordem original e agrupamos blocos vizinhos enquanto a soma continuar dentro do limite. Um heading inicia uma nova unidade, e um bloco estrutural que não cabe permanece inteiro para que a prática não corte uma tabela ou um JSON no meio.

In [10]:
def compose_structural_chunks(blocks, max_tokens):
    chunks = []
    current_blocks = []
    current_tokens = 0

    def flush():
        nonlocal current_blocks, current_tokens
        if not current_blocks:
            return
        chunks.append({
            "chunk_id": f"chunk-structural-{len(chunks) + 1:02d}",
            "blocks": current_blocks,
            "text": "\n\n".join(block["text"] for block in current_blocks),
            "token_count": current_tokens,
        })
        current_blocks = []
        current_tokens = 0

    for block in blocks:
        starts_new_unit = block["kind"] == "heading" and current_blocks
        exceeds_limit = current_blocks and current_tokens + block["token_count"] > max_tokens

        if starts_new_unit or exceeds_limit:
            flush()

        current_blocks.append(block)
        current_tokens += block["token_count"]

    flush()
    return chunks


structural_chunks = compose_structural_chunks(structural_blocks, MAX_CHUNK_TOKENS)

print(f"chunks finais: {len(structural_chunks)}")
for chunk in structural_chunks:
    kinds = ", ".join(block["kind"] for block in chunk["blocks"])
    print(f"{chunk['chunk_id']} | {chunk['token_count']} tokens | {kinds}")

chunks finais: 5
chunk-structural-01 | 113 tokens | heading, paragraph
chunk-structural-02 | 91 tokens | paragraph
chunk-structural-03 | 126 tokens | table
chunk-structural-04 | 98 tokens | paragraph, code
chunk-structural-05 | 90 tokens | paragraph


## 6. Inspecionando as unidades recuperáveis

Aqui a diferença para o corte por tamanho fica visível: tabela e código aparecem como blocos reconhecíveis. Quando existe espaço, o algoritmo agrupa blocos vizinhos; quando não existe, a fronteira respeita o limite do modelo sem destruir a unidade estrutural.

In [12]:
for chunk in structural_chunks:
    kinds = ", ".join(block["kind"] for block in chunk["blocks"])
    print(f"{chunk['chunk_id']} | {chunk['token_count']} tokens | {kinds}")
    print(chunk["text"])
    print("-" * 72)

chunk-structural-01 | 113 tokens | heading, paragraph
# 4.3 Timeouts e retentativas em webhooks

Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor.
Se a conexao expira antes de receber confirmacao, o provedor nao sabe se o evento
falhou antes de chegar, se foi processado parcialmente ou se a resposta se perdeu
no caminho de volta.
------------------------------------------------------------------------
chunk-structural-02 | 91 tokens | paragraph
Por isso, timeout nao deve ser tratado como erro definitivo. Em geral, ele entra
na mesma familia de falhas temporarias: o provedor registra a tentativa, agenda
uma nova entrega e preserva o mesmo identificador de evento para permitir
deduplicacao no consumidor.
------------------------------------------------------------------------
chunk-structural-03 | 126 tokens | table
| status | significado | acao recomendada |
| --- | --- | --- |
| 200 | evento recebido e persistido | encerrar entrega |
| 408 | consumidor 

## 7. Anexando metadados

A unidade recuperável não é apenas o texto composto. Ela também precisa carregar a origem global e a posição local que permitem filtrar, rastrear e reconstruir contexto depois da busca.

In [14]:
metadata_chunks = []

for order, chunk in enumerate(structural_chunks, start=1):
    first_block = chunk["blocks"][0]
    metadata = {
        **page_metadata,
        "chunk_id": chunk["chunk_id"],
        "strategy": "structural_token_aware",
        "order": order,
        "heading_path": first_block["heading_path"],
        "content_types": [block["kind"] for block in chunk["blocks"]],
        "token_count": chunk["token_count"],
        "source_file": str(DATA_PATH),
    }
    metadata_chunks.append({"text": chunk["text"], "metadata": metadata})

for item in metadata_chunks:
    metadata = item["metadata"]
    path = " > ".join(metadata["heading_path"])
    kinds = ", ".join(metadata["content_types"])
    print(
        f"{metadata['chunk_id']} | ordem {metadata['order']} | "
        f"{metadata['token_count']} tokens | {kinds} | {path}"
    )

chunk-structural-01 | ordem 1 | 113 tokens | heading, paragraph | 4.3 Timeouts e retentativas em webhooks
chunk-structural-02 | ordem 2 | 91 tokens | paragraph | 4.3 Timeouts e retentativas em webhooks
chunk-structural-03 | ordem 3 | 126 tokens | table | 4.3 Timeouts e retentativas em webhooks
chunk-structural-04 | ordem 4 | 98 tokens | paragraph, code | 4.3 Timeouts e retentativas em webhooks
chunk-structural-05 | ordem 5 | 90 tokens | paragraph | 4.3 Timeouts e retentativas em webhooks


## 8. Recuperando um candidato

A busca permanece simples de propósito. Ela apenas conta termos da query nos chunks já formados. O ponto é observar qual unidade estrutural chega ao retrieval, sem transformar a prática em uma aula de ranking ou de banco vetorial.

In [16]:
STOPWORDS = {"como", "com", "em", "de", "o", "a", "os", "as", "um", "uma"}

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9áéíóúãõç\s]", " ", text)
    return text

def tokenize_terms(text):
    return [term for term in re.findall(r"[a-z0-9áéíóúãõç]+", normalize(text)) if term not in STOPWORDS]

def search(query, items):
    query_terms = set(tokenize_terms(query))
    ranking = []

    for item in items:
        chunk_terms = tokenize_terms(item["text"])
        matches = sorted(query_terms.intersection(chunk_terms))
        score = sum(chunk_terms.count(term) for term in matches)
        ranking.append((score, matches, item))

    return sorted(ranking, key=lambda result: result[0], reverse=True)


query = "como lidar com timeout em webhooks?"
results = search(query, metadata_chunks)

print(f"query: {query}")
for position, (score, matches, item) in enumerate(results, start=1):
    metadata = item["metadata"]
    matched_terms = ", ".join(matches) if matches else "sem termos da query"
    print(f"#{position} {metadata['chunk_id']} score={score} ({matched_terms})")

best = results[0][2]
print("\nmelhor candidato:")
print(best["metadata"]["chunk_id"])
print(best["text"])

query: como lidar com timeout em webhooks?
#1 chunk-structural-01 score=1 (webhooks)
#2 chunk-structural-02 score=1 (timeout)
#3 chunk-structural-05 score=1 (timeout)
#4 chunk-structural-03 score=0 (sem termos da query)
#5 chunk-structural-04 score=0 (sem termos da query)

melhor candidato:
chunk-structural-01
# 4.3 Timeouts e retentativas em webhooks

Quando um provedor envia um webhook, ele espera uma resposta rapida do consumidor.
Se a conexao expira antes de receber confirmacao, o provedor nao sabe se o evento
falhou antes de chegar, se foi processado parcialmente ou se a resposta se perdeu
no caminho de volta.


O chunking estrutural preservou a hierarquia e as unidades reconhecíveis da página, mas não eliminou a restrição técnica. A estrutura orientou as fronteiras; o tokenizer do modelo decidiu até onde cada composição poderia crescer. Em uma prática semântica, os embeddings entrarão como outro sinal para escolher essas fronteiras.